<a href="https://colab.research.google.com/github/viswajithpanduga1607/Financial-Chatbot/blob/main/Financial_Chatbot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Cell 1: Install required libraries
!pip install -q gradio google-generativeai

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-cloud-spanner 3.71.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 5.29.6 which is incompatible.
google-cloud-storage-control 1.17.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 5.29.6 which is incompatible.
google-cloud-language 2.22.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 5.29.6 which is incompatible.
google-cloud-translate 3.28.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 5.29.6 which is incompatible.
google-cloud-bigquery-storage 2.43.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 5.29.6 which is incompatible.
tensorflow 2.21.0 requires protobuf<8.0.0,>=6.31.1, but you have protobuf 5.29.6 which is incompatible.
google-cloud-functions 1.25.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 5.29.6 which is incompatible.
grpc-google-iam

In [2]:
# Cell 2: Import libraries and authenticate
import os
import gradio as gr
import google.generativeai as genai
from google.colab import userdata

# Safely retrieve your API key from Colab Secrets
try:
    api_key = userdata.get('GEMINI_API_KEY')
    genai.configure(api_key=api_key)
    print("API Key successfully configured.")
except userdata.SecretNotFoundError:
    print("Error: Please add your GEMINI_API_KEY to the Colab Secrets tab.")

/tmp/ipykernel_12086/3516103584.py:4: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


API Key successfully configured.


In [3]:
# Cell 3: Configure the Model and System Prompt
# We use gemini-1.5-flash for rapid, cost-effective chat responses
model_name = "gemini-1.5-flash"

system_instruction = """
You are an expert, professional AI Financial Advisor.
Your goal is to help users with budgeting, savings, loans, investments, and financial planning.
Follow these rules strictly:
1. Always be professional, objective, and clear.
2. Use established financial frameworks when applicable (e.g., 50/30/20 budgeting rule, emergency fund basics, debt snowball/avalanche).
3. If a user asks a non-financial question, politely redirect them back to finance (e.g., "I specialize in finance. How can I help you with your budget or investments today?").
4. Always include a disclaimer that you are an AI and your advice should not replace a certified financial planner.
5. Format your responses with bullet points and clear headings for readability.
"""

# Initialize the model with the strict persona
model = genai.GenerativeModel(
    model_name=model_name,
    system_instruction=system_instruction
)

# We need a function to manage the chat history between Gradio and the Gemini API
def predict(message, history):
    # Convert Gradio's history format to Gemini's expected format
    gemini_history = []
    for user_msg, bot_msg in history:
        gemini_history.append({"role": "user", "parts": [user_msg]})
        gemini_history.append({"role": "model", "parts": [bot_msg]})

    # Start a chat session with the historical context
    chat = model.start_chat(history=gemini_history)

    # Send the new message and stream the response
    response = chat.send_message(message, stream=True)

    partial_message = ""
    for chunk in response:
        if chunk.text:
            partial_message += chunk.text
            yield partial_message

In [ ]:
# Cell 4: Build and Launch the Gradio Interface
import gradio as gr

# Custom CSS for a vintage-luxury aesthetic over standard SaaS interfaces
custom_theme = gr.themes.Soft().set(
    body_background_fill="#FBF9D1",
    block_background_fill="#E6CFA9",
    button_primary_background_fill="#9A3F3F",
    button_primary_background_fill_hover="#C1856D",
    button_primary_text_color="white",
    button_secondary_background_fill="#C1856D",
    button_secondary_text_color="white"
)

# Build the chat interface inside a gr.Blocks container and apply the theme here
with gr.Blocks(theme=custom_theme) as demo:
    gr.ChatInterface(
        fn=predict,
        title="Intelligent Financial Chatbot",
        description="Ask me about budgeting, saving, loans, or market trends. Powered by Gemini & Gradio.",
        examples=[
            "I make $4,000 a month. Can you create a 50/30/20 budget for me?",
            "What is the difference between a fixed and variable rate loan?",
            "How much will I have in 10 years if I invest $500 a month at a 7% annual return?",
            "What are some common signs of credit card fraud I should look out for?"
        ]
        # Removed undo_btn and clear_btn to resolve the TypeError
    )

# Launch the app
demo.launch(share=True, debug=True)

/tmp/ipykernel_12086/3851467757.py:16: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=custom_theme) as demo:


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://63a4a0430896226f44.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
